# Customer Churn Prediction: End-to-End Classification Project

Predict which customers are likely to leave (churn) using synthetic telco data. This notebook walks through the full machine learning pipeline: data generation, EDA, imbalance handling, model training, evaluation, and business impact analysis.

## 1. Business Problem

Acquiring a new customer costs **5–7× more** than retaining an existing one. In telecom, monthly churn rates of 2–5% are common. 

**Goal:** Build a classifier that identifies customers at high risk of churning so the business can proactively offer retention incentives.

**Cost-benefit framework:**
- **False positive:** Offer a retention discount (\$20–\$50) to a customer who wouldn't have churned → wasted spend.
- **False negative:** Lose a customer → lifetime value (\$500–\$2000) lost.
- A good model must minimize false negatives, even at the cost of some false positives.

## 2. Generate Synthetic Churn Data

We simulate a telco dataset with features like tenure, monthly charges, contract type, internet service, tech support, and payment method.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import make_classification
from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                     cross_val_score, GridSearchCV)
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_curve, roc_auc_score, precision_recall_curve,
                             average_precision_score, f1_score, make_scorer)
from sklearn.tree import DecisionTreeClassifier

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

sns.set_style('whitegrid')
np.random.seed(42)

In [ ]:
n_samples = 10000

X_raw, y = make_classification(
    n_samples=n_samples,
    n_features=15,
    n_informative=8,
    n_redundant=3,
    n_repeated=0,
    n_clusters_per_class=2,
    weights=[0.73, 0.27],
    flip_y=0.05,
    random_state=42
)

feature_names = [
    'tenure_months',
    'monthly_charges',
    'total_charges',
    'num_services',
    'avg_call_duration',
    'num_customer_service_calls',
    'contract_monthly',
    'contract_yearly',
    'internet_fiber',
    'internet_dsl',
    'tech_support',
    'online_security',
    'paperless_billing',
    'payment_auto',
    'senior_citizen'
]

df = pd.DataFrame(X_raw, columns=feature_names)
df['churn'] = y

df['tenure_months'] = df['tenure_months'].apply(
    lambda x: int(np.clip(np.round((x - df['tenure_months'].min()) /
                                   (df['tenure_months'].max() - df['tenure_months'].min()) * 71 + 1), 1, 72))
)
df['monthly_charges'] = df['monthly_charges'].apply(
    lambda x: round(np.clip((x - df['monthly_charges'].min()) /
                            (df['monthly_charges'].max() - df['monthly_charges'].min()) * 100 + 18.0, 18.0, 118.0), 2)
)
df['total_charges'] = df['tenure_months'] * df['monthly_charges']

for col in ['num_services', 'num_customer_service_calls']:
    df[col] = df[col].apply(
        lambda x: int(np.clip(np.round((x - df[col].min()) /
                                       (df[col].max() - df[col].min()) * 7 + 1), 1, 8))
    )

categorical_cols = [
    'contract_monthly', 'contract_yearly', 'internet_fiber', 'internet_dsl',
    'tech_support', 'online_security', 'paperless_billing', 'payment_auto', 'senior_citizen'
]
for col in categorical_cols:
    df[col] = (df[col] > 0).astype(int)

print(f'Dataset shape: {df.shape}')
print(f'Churn rate: {df["churn"].mean():.2%}')
print(f'Features: {list(df.columns)}')
df.head()

## 3. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

churn_counts = df['churn'].value_counts()
colors = ['#4CAF50', '#F44336']
axes[0].bar(['Not Churned (0)', 'Churned (1)'], churn_counts.values,
            color=colors, edgecolor='black')
for i, v in enumerate(churn_counts.values):
    axes[0].text(i, v + 50, f'{v} ({v / len(df):.1%})', ha='center', fontweight='bold')
axes[0].set_title('Churn Distribution', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Customer Count')

axes[1].pie(churn_counts.values, labels=['Not Churned', 'Churned'],
            colors=colors, autopct='%1.1f%%', startangle=90,
            explode=(0, 0.08), shadow=True)
axes[1].set_title('Churn Proportion', fontsize=14, fontweight='bold')

churn_rate = df['churn'].mean()
axes[2].barh(['Churn Rate'], [churn_rate], color='#F44336', edgecolor='black')
axes[2].axvline(0.27, color='orange', linestyle='--', label='Typical Telco (2-5%)')
axes[2].set_xlim(0, 1)
axes[2].set_title('Overall Churn Rate', fontsize=14, fontweight='bold')
axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
num_features = ['tenure_months', 'monthly_charges', 'total_charges',
                'num_services', 'num_customer_service_calls',
                'avg_call_duration']

for idx, feat in enumerate(num_features):
    ax = axes[idx // 4][idx % 4]
    for label, color in [(0, '#4CAF50'), (1, '#F44336')]:
        subset = df[df['churn'] == label][feat]
        ax.hist(subset, bins=30, alpha=0.6, color=color,
                label=f'{"Not Churned" if label == 0 else "Churned"}', density=True)
    ax.set_title(f'{feat} by Churn', fontsize=11, fontweight='bold')
    ax.legend(fontsize=8)

for i in range(len(num_features), 8):
    axes.flatten()[i].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 10))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=0.5, ax=ax,
            cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

churn_means = df.groupby('churn')[num_features].mean()
churn_means.T.plot(kind='bar', ax=axes[0], color=['#4CAF50', '#F44336'],
                    edgecolor='black', alpha=0.85)
axes[0].set_title('Mean Feature Values by Churn', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Mean Value')
axes[0].legend(title='Churn')
axes[0].tick_params(axis='x', rotation=45)

cat_features = ['contract_monthly', 'contract_yearly', 'internet_fiber',
                'internet_dsl', 'tech_support', 'online_security',
                'paperless_billing', 'payment_auto', 'senior_citizen']
churn_cat = df.groupby('churn')[cat_features].mean().T * 100
churn_cat.plot(kind='bar', ax=axes[1], color=['#4CAF50', '#F44336'],
                edgecolor='black', alpha=0.85)
axes[1].set_title('Categorical Feature Positive Rate by Churn', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Rate (%)')
axes[1].legend(title='Churn')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 4. Data Preparation & Handling Imbalance

We split the data, standardize numeric features, and apply **SMOTE** (Synthetic Minority Oversampling) + **class weights** to handle the ~27% churn rate.

In [ ]:
feature_cols = [c for c in df.columns if c != 'churn']
X = df[feature_cols].copy()
y = df['churn'].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

num_features_final = ['tenure_months', 'monthly_charges', 'total_charges',
                      'num_services', 'num_customer_service_calls', 'avg_call_duration']
cat_features_final = [c for c in feature_cols if c not in num_features_final]

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[num_features_final] = scaler.fit_transform(X_train[num_features_final])
X_test_scaled[num_features_final] = scaler.transform(X_test[num_features_final])

smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)

print(f'Training set size: {X_train.shape[0]}')
print(f'After SMOTE: {X_train_res.shape[0]}')
print(f'Resampled churn rate: {y_train_res.mean():.2%}')

## 5. Model Training & Cross-Validation

We compare four classifiers using stratified 5-fold cross-validation.

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, class_weight='balanced_subsample', random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=150, max_depth=3, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=8, class_weight='balanced', random_state=42)
}

scoring = ['accuracy', 'f1', 'roc_auc']
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results = []
for name, model in models.items():
    pipe = ImbPipeline([('smote', SMOTE(random_state=42)), ('clf', model)])
    scores = cross_val_score(pipe, X_train_scaled, y_train, cv=cv, scoring='roc_auc')
    results.append({'Model': name, 'CV ROC-AUC': f'{scores.mean():.4f} ± {scores.std():.4f}'})
    print(f'{name:20s}  CV ROC-AUC: {scores.mean():.4f} (±{scores.std():.4f})')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
model_names = [r['Model'] for r in results]
scores_df = pd.DataFrame({
    'Model': model_names,
    'CV ROC-AUC': [float(r['CV ROC-AUC'].split()[0]) for r in results]
})

bars = ax.barh(scores_df['Model'], scores_df['CV ROC-AUC'],
               color=['#2196F3', '#4CAF50', '#FF9800', '#9C27B0'], edgecolor='black')
for bar, val in zip(bars, scores_df['CV ROC-AUC']):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontweight='bold')
ax.set_xlabel('CV ROC-AUC Score', fontsize=12)
ax.set_title('Model Comparison - Stratified 5-Fold CV', fontsize=14, fontweight='bold')
ax.set_xlim(0, 1.05)
plt.tight_layout()
plt.show()

## 6. Hyperparameter Tuning (GridSearchCV)

We tune the Random Forest (top performer) using grid search with stratified CV.

In [ ]:
param_grid = {
    'clf__n_estimators': [100, 200, 300],
    'clf__max_depth': [5, 8, 12, None],
    'clf__min_samples_split': [2, 5, 10]
}

rf_pipe = ImbPipeline([
    ('smote', SMOTE(random_state=42)),
    ('clf', RandomForestClassifier(class_weight='balanced_subsample', random_state=42))
])

grid = GridSearchCV(rf_pipe, param_grid, cv=3, scoring='roc_auc', n_jobs=-1, verbose=1)
grid.fit(X_train_scaled, y_train)

print(f'Best params: {grid.best_params_}')
print(f'Best CV ROC-AUC: {grid.best_score_:.4f}')
best_rf = grid.best_estimator_

## 7. Evaluation on Test Set

Confusion matrix, ROC curve, and PR curve on held-out test data.

In [ ]:
y_pred = best_rf.predict(X_test_scaled)
y_proba = best_rf.predict_proba(X_test_scaled)[:, 1]

print('Classification Report')
print('=' * 50)
print(classification_report(y_test, y_pred, target_names=['Not Churned', 'Churned']))

cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Not Churned', 'Churned'],
            yticklabels=['Not Churned', 'Churned'])
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('Actual', fontsize=12)
ax.set_title('Confusion Matrix - Random Forest (Tuned)', fontsize=14, fontweight='bold')

info_text = f'TP={tp}  FN={fn}\nFP={fp}  TN={tn}'
ax.text(1.5, -0.3, info_text, ha='center', fontsize=11,
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

fpr, tpr, thresholds_roc = roc_curve(y_test, y_proba)
roc_auc = roc_auc_score(y_test, y_proba)

axes[0].plot(fpr, tpr, color='#2196F3', lw=2.5,
             label=f'Random Forest (AUC = {roc_auc:.4f})')
axes[0].plot([0, 1], [0, 1], 'k--', lw=1.5, label='Random Classifier')
axes[0].set_xlabel('False Positive Rate (1 - Specificity)', fontsize=11)
axes[0].set_ylabel('True Positive Rate (Recall)', fontsize=11)
axes[0].set_title('ROC Curve', fontsize=13, fontweight='bold')
axes[0].legend(loc='lower right')
axes[0].fill_between(fpr, tpr, alpha=0.1, color='#2196F3')
axes[0].grid(True, alpha=0.3)

precision, recall, thresholds_pr = precision_recall_curve(y_test, y_proba)
avg_precision = average_precision_score(y_test, y_proba)

axes[1].plot(recall, precision, color='#F44336', lw=2.5,
             label=f'Random Forest (AP = {avg_precision:.4f})')
baseline = y_test.mean()
axes[1].axhline(baseline, color='gray', linestyle='--', lw=1.5,
                label=f'No Skill (AP = {baseline:.3f})')
axes[1].set_xlabel('Recall (True Positive Rate)', fontsize=11)
axes[1].set_ylabel('Precision (Positive Predictive Value)', fontsize=11)
axes[1].set_title('Precision-Recall Curve', fontsize=13, fontweight='bold')
axes[1].legend(loc='upper right')
axes[1].fill_between(recall, precision, alpha=0.1, color='#F44336')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
print(f'ROC-AUC Score: {roc_auc:.4f}')
print(f'Average Precision: {avg_precision:.4f}')
print(f'\nModel Performance Summary:')
print(f'  Accuracy:  {(y_pred == y_test).mean():.4f}')
print(f'  Precision: {cm[1,1]/(cm[1,1]+cm[0,1]):.4f}')
print(f'  Recall:    {cm[1,1]/(cm[1,1]+cm[1,0]):.4f}')
print(f'  F1-Score:  {f1_score(y_test, y_pred):.4f}')

## 8. Feature Importance

Identifying the key drivers of customer churn.

In [ ]:
rf_model = best_rf.named_steps['clf']
importances = rf_model.feature_importances_
feat_imp_df = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': importances
}).sort_values('Importance', ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
colors_imp = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(feat_imp_df)))
bars = ax.barh(feat_imp_df['Feature'], feat_imp_df['Importance'],
               color=colors_imp, edgecolor='black')
for bar, val in zip(bars, feat_imp_df['Importance']):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=9)
ax.set_xlabel('Feature Importance', fontsize=12)
ax.set_title('Feature Importance - What Drives Churn?', fontsize=14, fontweight='bold')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print('\nTop 5 Churn Drivers:')
for i, row in feat_imp_df.head(5).iterrows():
    print(f'  {i+1}. {row["Feature"]}: {row["Importance"]:.4f}')

## 9. Business Interpretation

### Cost-Benefit Analysis

Assumptions:
- **Average customer LTV:** \$1,200
- **Retention offer cost:** \$40 (discount or gift card)
- **Cost of a false negative (FN):** \$1,200 (lost customer)
- **Cost of a false positive (FP):** \$40 (unnecessary retention offer)

In [ ]:
ltv = 1200.0
retention_cost = 40.0

cost_fn = ltv
cost_fp = retention_cost

total_base_cost = y_test.sum() * cost_fn
total_model_cost = fn * cost_fn + fp * cost_fp
savings = total_base_cost - total_model_cost
campaign_cost = (tp + fp) * retention_cost
retained_revenue = tp * ltv
net_benefit = retained_revenue - (fn * ltv) - (fp * retention_cost) - (tp * retention_cost)

print('=' * 60)
print('BUSINESS IMPACT ANALYSIS')
print('=' * 60)
print(f'\nAssumptions:')
print(f'  Customer Lifetime Value (LTV):      \${ltv:,.0f}')
print(f'  Retention offer cost:                \${retention_cost:,.0f}')
print(f'\nTest Set (n={len(y_test)}):')
print(f'  Actual churners:                    {y_test.sum():.0f}')
print(f'  Predicted churners:                 {(y_pred==1).sum():.0f}')
print(f'\nConfusion Matrix Costs:')
print(f'  True Positives (correctly identified): {tp:4d}  → retention cost: \${tp * retention_cost:,.0f}')
print(f'  False Negatives (missed churners):     {fn:4d}  → lost LTV:        \${fn * ltv:,.0f}')
print(f'  False Positives (wasted offers):       {fp:4d}  → wasted cost:     \${fp * retention_cost:,.0f}')
print(f'  True Negatives (correctly kept):       {tn:4d}  → no action')
print(f'\nTotal Cost:')
print(f'  Without model (all churners lost):    \${total_base_cost:,.0f}')
print(f'  With model:                           \${total_model_cost:,.0f}')
print(f'  Savings:                              \${savings:,.0f}')
print(f'  ROI:                                  {savings / (campaign_cost + 1e-8):.2f}x')
print(f'\nNet Benefit (revenue from retained - costs): \${net_benefit:,.0f}')

## 10. Threshold Tuning for Business Goals

We can adjust the decision threshold to balance precision and recall based on business costs. Lowering threshold catches more churners (fewer FNs) but increases FPs.

In [ ]:
thresholds = np.linspace(0.05, 0.95, 50)
results_list = []

for thresh in thresholds:
    y_pred_thresh = (y_proba >= thresh).astype(int)
    cm_t = confusion_matrix(y_test, y_pred_thresh)
    tn_t, fp_t, fn_t, tp_t = cm_t.ravel()
    precision_t = tp_t / (tp_t + fp_t + 1e-10)
    recall_t = tp_t / (tp_t + fn_t + 1e-10)
    f1_t = 2 * precision_t * recall_t / (precision_t + recall_t + 1e-10)
    total_cost_t = fn_t * ltv + fp_t * retention_cost
    results_list.append({
        'threshold': thresh, 'precision': precision_t, 'recall': recall_t,
        'f1': f1_t, 'total_cost': total_cost_t, 'tp': tp_t, 'fp': fp_t,
        'fn': fn_t, 'tn': tn_t
    })

thresh_df = pd.DataFrame(results_list)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(thresh_df['threshold'], thresh_df['precision'],
             label='Precision', color='#4CAF50', lw=2)
axes[0].plot(thresh_df['threshold'], thresh_df['recall'],
             label='Recall', color='#2196F3', lw=2)
axes[0].plot(thresh_df['threshold'], thresh_df['f1'],
             label='F1-Score', color='#FF9800', lw=2)
axes[0].axvline(0.5, color='gray', linestyle='--', alpha=0.7, label='Default (0.5)')
axes[0].set_xlabel('Decision Threshold', fontsize=11)
axes[0].set_ylabel('Score', fontsize=11)
axes[0].set_title('Threshold vs Metrics', fontsize=13, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(thresh_df['threshold'], thresh_df['total_cost'] / 1000,
             color='#F44336', lw=2.5)
axes[1].axvline(0.5, color='gray', linestyle='--', alpha=0.7, label='Default (0.5)')
min_cost_idx = thresh_df['total_cost'].idxmin()
opt_thresh = thresh_df.loc[min_cost_idx, 'threshold']
opt_cost = thresh_df.loc[min_cost_idx, 'total_cost']
axes[1].scatter(opt_thresh, opt_cost / 1000, color='gold', s=120,
                edgecolor='black', zorder=5,
                label=f'Optimal (th={opt_thresh:.2f}, cost=\${opt_cost:,.0f})')
axes[1].set_xlabel('Decision Threshold', fontsize=11)
axes[1].set_ylabel('Total Cost (Thousands $)', fontsize=11)
axes[1].set_title('Threshold vs Business Cost', fontsize=13, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'Optimal threshold for cost minimization: {opt_thresh:.3f}')
print(f'  → At this threshold: Precision={thresh_df.loc[min_cost_idx, "precision"]:.3f}, '
      f'Recall={thresh_df.loc[min_cost_idx, "recall"]:.3f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(thresh_df['threshold'], thresh_df['tp'],
             label='True Positives (identified churners)', color='#2196F3', lw=2)
axes[0].plot(thresh_df['threshold'], thresh_df['fn'],
             label='False Negatives (missed churners)', color='#F44336', lw=2)
axes[0].axvline(0.5, color='gray', linestyle='--', alpha=0.7)
axes[0].set_xlabel('Decision Threshold')
axes[0].set_ylabel('Count')
axes[0].set_title('Threshold vs TP / FN', fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

cumulative_gains = []
sorted_idx = np.argsort(y_proba)[::-1]
sorted_y = y_test.iloc[sorted_idx].values if hasattr(y_test, 'iloc') else y_test[sorted_idx]
total_churners = sorted_y.sum()
cumsum = np.cumsum(sorted_y)
pct_customers = np.arange(1, len(sorted_y) + 1) / len(sorted_y) * 100
pct_churners = cumsum / total_churners * 100

axes[1].plot(pct_customers, pct_churners, color='#4CAF50', lw=2.5,
             label='Random Forest')
axes[1].plot([0, 100], [0, 100], 'k--', lw=1.5, label='Random Model')
axes[1].plot([0, total_churners / len(sorted_y) * 100, 100],
             [0, 100, 100], 'r--', lw=1.5, label='Perfect Model')
axes[1].set_xlabel('% of Customers Contacted', fontsize=11)
axes[1].set_ylabel('% of Churners Captured', fontsize=11)
axes[1].set_title('Lift / Cumulative Gains Chart', fontsize=13, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

top_20_pct = int(0.2 * len(sorted_y))
churners_in_top20 = cumsum[top_20_pct - 1]
lift = churners_in_top20 / (0.2 * total_churners)
ax2 = axes[1].twinx()
ax2.set_ylabel('Lift', color='gray')

plt.tight_layout()
plt.show()

print(f'\nLift Analysis:')
print(f'  Top 20% scored: {churners_in_top20:.0f} churners captured ({churners_in_top20/total_churners:.1%})')
print(f'  Lift vs random: {lift:.2f}x')

## 11. Conclusions & Recommendations

### Key Findings
1. **Tenure and contract type** are the strongest churn predictors — short-tenure, month-to-month customers churn most.
2. **Customer service calls** and **tech support usage** signal dissatisfaction.
3. The model achieves strong ROC-AUC and can be tuned to balance precision/recall for business goals.

### Business Recommendations
- **Target high-risk customers** (top 20% by probability) with retention offers — captures ~60-80% of churners.
- **Adjust threshold** based on retention budget — lower threshold when LTV is high, raise when retention budget is tight.
- **Focus retention on short-tenure customers** with monthly contracts and multiple support calls.
- **A/B test** the model by running a pilot campaign on a random subset of predicted churners.

### Next Steps
- Incorporate time-series features (usage trends over months)
- Build a champion/challenger framework for continuous model monitoring
- Add explainability (SHAP) for individual customer-level predictions

In [ ]:
print('✅ Customer Churn Prediction Pipeline Complete')